# Ukraine Shahed Drone — Weather & Interception Rate Analysis

**Research question:** Does inclement night-time weather reduce Ukraine's ability to intercept Russian Shahed-136/Geran-2 long-range attack drones?

**Secondary question:** Does Russia deliberately increase launch volumes on adverse-weather nights, suggesting deliberate exploitation of Ukrainian sensor limitations?

**Data sources:**
- Attack data: Petro Ivaniuk Kaggle dataset (`pityfm/massive-missile-attacks-on-ukraine`)
- Weather data: Open-Meteo Historical Weather API (free, no key required)

**Analytical rationale:** Shaheds are subsonic, low-altitude loitering munitions. Ukrainian interception relies on a layered mix of radar-guided systems, optically/thermally guided MANPADS, mobile fire groups, and fighter intercepts. Cloud cover and reduced visibility degrade optical/thermal sensors and complicate visual identification. Rain and fog also affect radar cross-section tracking at low altitudes. Night-time conditions are used throughout as Shahed attacks overwhelmingly occur between 20:00–06:00 local time.

---

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from pathlib import Path

try:
    import statsmodels.formula.api as smf
    HAS_STATSMODELS = True
    print('statsmodels available — full regression enabled')
except ImportError:
    HAS_STATSMODELS = False
    print('statsmodels not installed — using scipy correlations only')
    print('Install with: pip install statsmodels')

DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../outputs/charts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete')

## 1. Classify Attacks

Load the raw Kaggle CSV and classify by weapon type. Isolate Shahed-136/Geran-2 rows.

> **Prerequisites:** Place `missile_attacks_daily.csv` in the `data/` directory first.  
> Download from: https://www.kaggle.com/datasets/pityfm/massive-missile-attacks-on-ukraine

In [ ]:
from classify_attacks import load_and_classify, build_shahed_daily, aggregate_by_date

raw_path = DATA_DIR / 'missile_attacks_daily.csv'
df_raw   = load_and_classify(raw_path)
df_raw.head(3)

In [ ]:
# Weapon class distribution
print('Weapon class distribution (rows):')
print(df_raw['weapon_class'].value_counts())
print()

# Launched by class
by_class = df_raw.groupby('weapon_class')['launched'].sum().sort_values(ascending=False)
print('Total launched by class:')
print(by_class)

In [ ]:
shahed_daily = build_shahed_daily(df_raw)
shahed_daily.head(5)

In [ ]:
# Quick check: interception rate distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(shahed_daily['interception_rate'] * 100, bins=20, 
        color='#005BBB', edgecolor='white', linewidth=0.8)
ax.axvline(shahed_daily['interception_rate'].mean() * 100, 
           color='#FFD700', linewidth=2, linestyle='--',
           label=f"Mean: {shahed_daily['interception_rate'].mean():.1%}")
ax.set_xlabel('Shahed Interception Rate (%)')
ax.set_ylabel('Number of Attack Days')
ax.set_title('Distribution of Daily Shahed Interception Rates')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Fetch Weather Data

Pull historical weather from Open-Meteo for each attack date, across four Ukrainian cities.

**Night-time window used:** 20:00–06:00 local (Europe/Kyiv) — reflects actual Shahed arrival windows.

Cities: Kyiv, Kharkiv, Odesa, Zaporizhzhia (national average computed).

In [ ]:
from fetch_weather import fetch_all_locations

attack_dates = shahed_daily['date'].dt.strftime('%Y-%m-%d').tolist()
print(f'{len(attack_dates)} unique Shahed attack dates to fetch weather for')
print(f'Range: {min(attack_dates)} → {max(attack_dates)}')

In [ ]:
# This will make API calls to Open-Meteo (~4 requests, takes ~10 seconds)
weather_path = DATA_DIR / 'weather_by_date.csv'

if weather_path.exists():
    print('Loading cached weather data...')
    weather_df = pd.read_csv(weather_path, parse_dates=['date'])
    print(f'Loaded {len(weather_df)} rows')
else:
    print('Fetching from Open-Meteo API...')
    weather_df = fetch_all_locations(attack_dates)
    weather_df.to_csv(weather_path, index=False)
    print(f'Saved to {weather_path}')

weather_df.head(3)

In [ ]:
# Available weather columns
weather_cols = [c for c in weather_df.columns 
                if c.startswith('avg_') or c == 'weather_category']
print('National average weather variables:')
for c in weather_cols:
    print(f'  {c}')

## 3. Merge & Assign Weather Buckets

In [ ]:
from analysis import assign_weather_bucket

# Merge
df = shahed_daily.merge(weather_df, on='date', how='inner')
print(f'Merged: {len(df)} attack days with weather data')

# Assign buckets
df = assign_weather_bucket(df)

print('\nWeather bucket distribution:')
print(df['weather_bucket'].value_counts())

## 4. Bucket Analysis: Interception Rate by Weather Condition

In [ ]:
from analysis import bucket_analysis

summary = bucket_analysis(df)
print(summary[['weather_bucket','n_days','mean_rate','median_rate','std_rate','ci95_low','ci95_high']].to_string(index=False))

In [ ]:
from analysis import (fig1_bucket_bars, fig2_scatter_cloud, 
                       fig3_scatter_visibility, fig5_time_trend,
                       fig6_launch_volume_by_weather)

fig1_bucket_bars(df, summary)

## 5. Scatter Analysis: Individual Weather Variables

In [ ]:
fig2_scatter_cloud(df)

In [ ]:
fig3_scatter_visibility(df)

## 6. Regression Analysis

In [ ]:
from analysis import run_regression

model = run_regression(df)

## 7. Time Trend + Deliberate Exploitation Test

In [ ]:
fig5_time_trend(df)

In [ ]:
# Key question: does Russia launch MORE drones on adverse weather nights?
# If yes, this is evidence of deliberate sensor-degradation exploitation.
fig6_launch_volume_by_weather(df)

## 8. Summary of Findings

In [ ]:
clear_rate   = df[df['weather_bucket'] == 'CLEAR']['interception_rate']
overcast_rate= df[df['weather_bucket'] == 'OVERCAST']['interception_rate']
adverse_rate = df[df['weather_bucket'] == 'ADVERSE']['interception_rate']

clear_vol   = df[df['weather_bucket'] == 'CLEAR']['launched']
adverse_vol = df[df['weather_bucket'] == 'ADVERSE']['launched']

print('═' * 55)
print('  FINDINGS SUMMARY')
print('═' * 55)
print()
print(f'Overall mean interception rate:  {df["interception_rate"].mean():.1%}')
print(f'Overall median:                  {df["interception_rate"].median():.1%}')
print()
print('By weather condition:')
if len(clear_rate):   print(f'  Clear nights:    {clear_rate.mean():.1%}  (n={len(clear_rate)} days)')
if len(overcast_rate):print(f'  Overcast nights: {overcast_rate.mean():.1%}  (n={len(overcast_rate)} days)')
if len(adverse_rate): print(f'  Adverse nights:  {adverse_rate.mean():.1%}  (n={len(adverse_rate)} days)')
print()
if len(clear_rate) and len(adverse_rate):
    delta = (clear_rate.mean() - adverse_rate.mean()) * 100
    direction = 'LOWER' if delta > 0 else 'HIGHER'
    t, p = stats.ttest_ind(clear_rate, adverse_rate)
    print(f'  Δ Clear→Adverse: {abs(delta):.1f} percentage points {direction}')
    print(f'  t-test p-value:  {p:.4f} ({"significant" if p < 0.05 else "not significant"} at α=0.05)')
print()
print('Deliberate exploitation test (launch volume):')
if len(clear_vol) and len(adverse_vol):
    t2, p2 = stats.ttest_ind(adverse_vol, clear_vol)
    direction2 = 'MORE' if adverse_vol.mean() > clear_vol.mean() else 'FEWER'
    print(f'  Russia launches {direction2} Shaheds on adverse nights')
    print(f'  Clear mean:   {clear_vol.mean():.1f} drones/night')
    print(f'  Adverse mean: {adverse_vol.mean():.1f} drones/night')
    print(f'  p-value:      {p2:.4f}')

---
## Notes on Limitations

1. **Weather averaging** — using a national average across four cities approximates but does not capture the actual weather along specific flight corridors (which vary by launch origin and target).
2. **Reported vs actual interceptions** — Ukrainian MoD figures may undercount or overcount depending on reporting incentives.
3. **Confounders not fully controlled** — improving Ukrainian AD capacity over time, Western system deliveries (IRIS-T, Patriot, NASAMS), seasonal variation, and wave size all affect interception rates independently of weather.
4. **Causal direction** — this analysis shows correlation; it cannot confirm whether Russia *deliberately selects* adverse weather windows or whether weather is incidental.
5. **Data lag** — the Kaggle dataset updates periodically; check for the latest version before drawing conclusions about recent months.
